# Secbutylamine

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
secbutylamine,73.1368,2.89794,3.560205164,236.6192571,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
secbutylamine,H,secbutylamine,e,2000.0,0.03
"""

model = PCSAFT(["secbutylamine"], userlocations = [like_parameter, assoc_parameter])

print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_secbutylamine.csv")
fix_line_endings("rhol_secbutylamine.csv")

Fixed: satp_secbutylamine.csv
Fixed: rhol_secbutylamine.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

760.2060638086308
495.6542138582546


In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1000.0,
        :guess   => 750.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 1.0,
        :guess   => 0.05
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 1000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 750.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.05, :lower => 0.001)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_secbutylamine.csv",
        "satp_secbutylamine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.007933369842282495


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([629.6257826756328, 0.10454082707252758], PCSAFT{BasicIdeal, Float64}("secbutylamine"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_secbutylamine.csv",   my_saturation_p)


=== AAD: satp_secbutylamine.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
299.8500    25265.000000  25303.778714  0.1535  
304.4400    30864.000000  30882.578343  0.0602  
313.8100    45356.000000  45374.669479  0.0412  
319.4600    56449.000000  56472.199380  0.0411  
325.0500    69487.000000  69489.632268  0.0038  
331.0900    86046.000000  86127.675649  0.0949  
335.2900    99432.000000  99446.912995  0.0150  
AARD = 0.0585%


0.05852134002897017

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_secbutylamine.csv",   my_liquid_density)


=== AAD: rhol_secbutylamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


298.1500    718.000000    724.866846    0.9564  
303.1500    713.100000    720.039245    0.9731  
308.1500    708.100000    715.180816    1.0000  
313.1500    703.100000    710.287396    1.0222  
318.1500    698.200000    705.354741    1.0247  
323.1500    693.200000    700.378512    1.0356  
AARD = 1.0020%


1.0020025186606725